In [1]:
import pandas as pd
df = pd.read_csv('news.csv')
dataset = df.drop("Unnamed: 0", axis=1)
dataset

,title,text,label
0,You Can Smell Hillary’s Fear,"Daniel Greenfield, a Shillman Journalism Fello...",FAKE
1,Watch The Exact Moment Paul Ryan Committed Pol...,Google Pinterest Digg Linkedin Reddit Stumbleu...,FAKE
2,Kerry to go to Paris in gesture of sympathy,U.S. Secretary of State John F. Kerry said Mon...,REAL
3,Bernie supporters on Twitter erupt in anger ag...,"— Kaydee King (@KaydeeKing) November 9, 2016 T...",FAKE
4,The Battle of New York: Why This Primary Matters,It's primary day in New York and front-runners...,REAL
...,...,...,...
6330,State Department says it can't find emails fro...,The State Department told the Republican Natio...,REAL
6331,The ‘P’ in PBS Should Stand for ‘Plutocratic’ ...,The ‘P’ in PBS Should Stand for ‘Plutocratic’ ...,FAKE
6332,Anti-Trump Protesters Are Tools of the Oligarc...,Anti-Trump Protesters Are Tools of the Oligar...,FAKE
6333,"In Ethiopia, Obama seeks progress on peace, se...","ADDIS ABABA, Ethiopia —President Obama convene...",REAL


In [2]:
x = dataset['text']
y = dataset['label']

from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.33, random_state=53, stratify=y)

In [3]:
# List of vectorizers to test
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer, HashingVectorizer
vectorizers = {
    "CountVectorizer": CountVectorizer(stop_words='english'),
    "TfidfVectorizer": TfidfVectorizer(stop_words='english'),
    "HashingVectorizer": HashingVectorizer(stop_words='english', alternate_sign=False)
}

In [4]:
# Model training 
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import PassiveAggressiveClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn import metrics


In [5]:
classifiers = {
    "MultinomialNB": MultinomialNB(),
    "PassiveAggressive": PassiveAggressiveClassifier(max_iter=1000, random_state=42),
    "DecisionTree": DecisionTreeClassifier(random_state=42),
    "RandomForest": RandomForestClassifier(n_estimators=100, random_state=42)
}

In [6]:
results = []

for v_name, vectorizer in vectorizers.items():
    for c_name, classifier in classifiers.items():
        text_clf = Pipeline([
            ('vectorizer', vectorizer),
            ('classifier', classifier)
        ])
        text_clf.fit(x_train, y_train)
        y_pred = text_clf.predict(x_test)
        accuracy = metrics.accuracy_score(y_test, y_pred)
        results.append({
            'Vectorizer': v_name,
            'Classifier': c_name,
            'Accuracy': accuracy
        })
        print(f"{v_name} + {c_name}: {accuracy:.3f}")

CountVectorizer + MultinomialNB: 0.880
CountVectorizer + PassiveAggressive: 0.896
CountVectorizer + DecisionTree: 0.796
CountVectorizer + RandomForest: 0.896
TfidfVectorizer + MultinomialNB: 0.826
TfidfVectorizer + PassiveAggressive: 0.929
TfidfVectorizer + DecisionTree: 0.799
TfidfVectorizer + RandomForest: 0.898
HashingVectorizer + MultinomialNB: 0.802
HashingVectorizer + PassiveAggressive: 0.918
HashingVectorizer + DecisionTree: 0.806
HashingVectorizer + RandomForest: 0.864


In [7]:
results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by='Accuracy', ascending=False).reset_index(drop=True)
print("\nTop 3 performing model combinations:\n")
print(results_df.head(3))



Top 3 performing model combinations:

          Vectorizer         Classifier  Accuracy
0    TfidfVectorizer  PassiveAggressive  0.928742
1  HashingVectorizer  PassiveAggressive  0.918221
2    TfidfVectorizer       RandomForest  0.897657
